# 07. LOT 240061 · SPHC

## 여러 온도의 초기 상승과 SPHC 안의 출력 차이

관측: **2024-06-15 01:04:00 ~ 2024-06-15 04:49:23**

1. 가열과 과시효 온도가 초기에 함께 오르는가?
2. 뒤쪽에서도 OP-Z4가 다른 SPHC보다 높은가?
3. 초기 상승을 이유로 거의 전체 LOT를 제외하는 것이 타당한가?

## 읽기 전에


원본 온도·조업이벤트·정비이력·정기수리·부품 CSV에서 독립 실행한다. 다른 노트북이나 가공 CSV를 먼저 실행할 필요가 없다. 센서마다 1분 내 유효 관측이 30개 이상일 때만 평균을 사용하고, 부족한 분과 LOT 사이 공백은 보간하지 않는다. 원시 그림도 누락 시각을 이어 붙이지 않는다.

자료는 교육용 재구성 데이터다. 데이터사전에서 온도 ℃와 출력 % 및 공정 구역 배정은 **가정**이고, CP-Z4의 실제 단위는 **확인되지 않았다**. OP의 0은 실제 제어출력으로 취급한다. CP-Z4M은 0이 아니라 작은 스케일의 중복 정보이므로 이 검토의 초점 센서에서 제외했다. 강종에 따른 출력 수준 이동도 검증된 실제 제철소 설정값을 뜻하지 않는다.

표의 기준은 원시 관측과 1분 평균을 구분해 표시했다. 표의 13:29는 13:29:00~59 구간의 평균이며, 곡선은 그 분의 가운데(13:29:30)에 표시한다. 해당 분이 끝난 뒤 계산할 수 있는 값이다. 아래 본문 표는 이번 원자료 실행에서 계산한 값이며, 원자료가 바뀌면 코드의 재실행 결과를 다시 확인한다. 이 노트북은 고장 정답을 만들거나 모델을 학습하지 않는다.

In [ ]:
LOT = 240061
NOTEBOOK_NAME = '07_LOT_240061'
PLOT_TITLE = '여러 온도의 초기 상승과 SPHC 안의 출력 차이'
FOCUS = ['TC-Z3', 'TC-Z4', 'TC-Z7', 'OP-Z4', 'CP-Z4']
COMPARE = ['TC-Z4', 'OP-Z4']
ZOOM_START = '2024-06-15 01:04'
ZOOM_END = '2024-06-15 02:45'
extra_tables = []

# 팀 공통 경로·인코딩·한글 폰트 설정입니다.
import os
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager, rcParams
import matplotlib.dates as mdates
from IPython.display import display

# 이 하위 폴더, notebooks 폴더, 저장소 루트에서 모두 실행할 수 있습니다.
for candidate in [os.path.join('..', '..', 'data'), os.path.join('..', 'data'), 'data']:
    if os.path.isfile(os.path.join(candidate, 'T-CR1-CAL01_온도.csv')):
        DATA_DIR = candidate
        break
else:
    raise FileNotFoundError('저장소의 data 폴더에 원본 CSV를 넣어 주세요.')

ROOT = Path(DATA_DIR).resolve().parent
FIG_DIR = os.path.join(ROOT, 'figures', '07_lot_anomaly_review_namelesspark', NOTEBOOK_NAME)
os.makedirs(FIG_DIR, exist_ok=True)
TEMP_CSV = os.path.join(DATA_DIR, 'T-CR1-CAL01_온도.csv')
EVENT_CSV = os.path.join(DATA_DIR, 'G-02_조업이벤트.csv')
REPAIR_CSV = os.path.join(DATA_DIR, 'G-01_정기수리캘린더.csv')
PART_CSV = os.path.join(DATA_DIR, 'G-01_부품마스터.csv')
MAINT_CSV = os.path.join(DATA_DIR, 'G-02_정비이력.csv')
PARTCHG_CSV = os.path.join(DATA_DIR, 'G-02_부품교체이력.csv')
ENC = 'utf-8-sig'

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)
installed = {f.name for f in font_manager.fontManager.ttflist}
for cand in ['Malgun Gothic', 'AppleGothic', 'NanumGothic', 'NanumBarunGothic', 'Noto Sans CJK KR']:
    if cand in installed:
        rcParams['font.family'] = cand
        break
else:
    print('경고: 한글 폰트를 찾지 못했습니다.')
rcParams['axes.unicode_minus'] = False
rcParams['figure.dpi'] = 110
rcParams['axes.spines.top'] = False
rcParams['axes.spines.right'] = False

## 원자료에서 LOT와 강종, 유효 1분 관측 만들기

LOT 시작 시각과 CR1 강종변경 시각을 연결한다. 센서별 관측 수를 따로 세어, 결측이 많은 센서의 분을 잘못 유효 처리하지 않는다.

In [ ]:
raw = pd.read_csv(TEMP_CSV, encoding=ENC, parse_dates=['MEAS_DT'])
events = pd.read_csv(EVENT_CSV, encoding=ENC, parse_dates=['EVT_DT'])
maintenance = pd.read_csv(MAINT_CSV, encoding=ENC, parse_dates=['DETC_DT', 'STRT_DT', 'END_DT'])
repair = pd.read_csv(REPAIR_CSV, encoding=ENC, parse_dates=['STRT_DT', 'END_DT'])
parts = pd.read_csv(PART_CSV, encoding=ENC)
part_changes = pd.read_csv(PARTCHG_CSV, encoding=ENC)

sensor_columns = [c for c in raw.columns if c not in ['LOT_NO', 'MEAS_DT', 'CP-Z4M']]
heat_columns = ['TC-Z3', 'TC-Z4', 'TC-Z5', 'TC-Z6']
lot_bounds = raw.groupby('LOT_NO').MEAS_DT.agg(start='min', end='max')
grade_events = events.loc[events.PLANT_CD.eq('CR1') & events.EVT_TYPE.eq('강종변경')]
# 연결이 중복되면 LOT 수가 늘어나므로 일대일 대응을 확인합니다.
lot_bounds = lot_bounds.reset_index()
lot_bounds = lot_bounds.merge(
    grade_events[['EVT_DT', 'AFT_VAL']],
    left_on='start',
    right_on='EVT_DT',
    how='left',
    validate='one_to_one',
)
lot_bounds = lot_bounds.set_index('LOT_NO')
lot_bounds = lot_bounds.rename(columns={'AFT_VAL': 'grade'})

# LOT별 1분 칸을 유지합니다. 관측 30개 미만인 센서 값은 비워 둡니다.
minute_by_lot = {}
count_by_lot = {}
for lot_no, group in raw.groupby('LOT_NO', sort=True):
    series = group.sort_values('MEAS_DT').set_index('MEAS_DT')
    means = series[sensor_columns].resample('1min').mean()
    counts = series[sensor_columns].resample('1min').count()
    means = means.mask(counts.lt(30))
    means['HEAT_MEAN'] = means[heat_columns].mean(axis=1, skipna=False)
    minute_by_lot[lot_no] = means
    count_by_lot[lot_no] = counts

raw_lot = raw.loc[raw.LOT_NO.eq(LOT)].sort_values('MEAS_DT').set_index('MEAS_DT')
minute = minute_by_lot[LOT]
lot_start = lot_bounds.loc[LOT, 'start']
lot_end = lot_bounds.loc[LOT, 'end']
grade = lot_bounds.loc[LOT, 'grade']
same_grade_lots = lot_bounds.index[lot_bounds.grade.eq(grade)].tolist()

## 이 검토에 실제 사용한 관측 수

| 센서 | 원시 유효 관측 | 원시 결측 | 전체 1분 칸 | 30개 이상인 분 | 30개 미만인 분 |
| --- | --- | --- | --- | --- | --- |
| TC-Z3 | 13299 | 0 | 226 | 225 | 1 |
| TC-Z4 | 13299 | 0 | 226 | 225 | 1 |
| TC-Z7 | 13299 | 0 | 226 | 225 | 1 |
| OP-Z4 | 13299 | 0 | 226 | 225 | 1 |
| CP-Z4 | 13299 | 0 | 226 | 225 | 1 |

아래 범위 표는 비교의 배경이다. 센서 단위가 다르므로 숫자의 크기를 센서끼리 직접 비교하지 않는다.

| 센서 | 1분 최솟값 | 1분 중앙값 | 1분 최댓값 | 원시 최솟값 | 원시 최댓값 |
| --- | --- | --- | --- | --- | --- |
| TC-Z3 | 813.764 | 857.303 | 870.57 | 812.657 | 870.704 |
| TC-Z4 | 816.729 | 859.339 | 861.428 | 814.268 | 861.52 |
| TC-Z7 | 313.904 | 329.707 | 330.067 | 313.038 | 330.161 |
| OP-Z4 | 46.417 | 54.291 | 62.457 | 43.4583 | 64.2358 |
| CP-Z4 | 0.442999 | 0.449998 | 0.46699 | 0.439914 | 0.4709 |

In [ ]:
sampling_rows = []

def readable_table(table):
    # CP의 작은 변화는 보존하고 온도·출력 표의 불필요한 소수 자릿수는 줄입니다.
    shown = table.astype(object).copy()
    for row_index, row in table.iterrows():
        sensor = str(row.get('센서', ''))
        for column, value in row.items():
            if isinstance(value, (float, np.floating)) and pd.notna(value):
                digits = 3
                if sensor.startswith('CP-'):
                    digits = 6
                elif sensor.startswith('OP-') and str(column).startswith('원시'):
                    digits = 4
                shown.at[row_index, column] = f'{value:.{digits}f}'.rstrip('0').rstrip('.')
    return shown

focus_rows = []
for sensor in FOCUS:
    source = raw_lot[sensor]
    means = minute[sensor]
    counts = count_by_lot[LOT][sensor]
    sampling_rows.append({
        '센서': sensor,
        '원시 유효 관측': int(source.notna().sum()),
        '원시 결측': int(source.isna().sum()),
        '전체 1분 칸': len(means),
        '30개 이상인 분': int(counts.ge(30).sum()),
        '30개 미만인 분': int(counts.lt(30).sum()),
    })
    focus_rows.append({
        '센서': sensor,
        '1분 최솟값': means.min(),
        '1분 중앙값': means.median(),
        '1분 최댓값': means.max(),
        '원시 최솟값': source.min(),
        '원시 최댓값': source.max(),
    })
sampling_table = pd.DataFrame(sampling_rows)
focus_table = pd.DataFrame(focus_rows)
display(sampling_table)
display(readable_table(focus_table))

## 같은 강종과 실제 정비 기록의 비교 준비

아래 비교값은 관측한 자료를 요약한 수치다. 미래 자료를 포함할 수 있으므로 학습 입력의 기준값으로 그대로 사용하지 않는다.

In [ ]:
# 같은 강종도 LOT별로 보여 줍니다. 긴 LOT에 가중된 강종 평균은 만들지 않습니다.
comparison_rows = []
comparison_values = {}
for lot_no in same_grade_lots:
    data = minute_by_lot[lot_no]
    scope = '관측 전체'
    if grade == 'SPCEN' and lot_no == 240114:
        data = data.loc[data.index < pd.Timestamp('2024-11-04 20:10')]
        scope = '20:10 이전: 후반 하강 전'
    comparison_values[lot_no] = data
    for sensor in COMPARE:
        series = data[sensor].dropna()
        comparison_rows.append({
            'LOT': lot_no,
            '시작일': lot_bounds.loc[lot_no, 'start'].strftime('%Y-%m-%d'),
            '비교 범위': scope,
            '센서': sensor,
            '유효 분': len(series),
            '25%': series.quantile(.25),
            '중앙값': series.median(),
            '75%': series.quantile(.75),
        })
comparison_table = pd.DataFrame(comparison_rows)

In [ ]:
# 같은 설비의 실제 기록만 연결합니다. 가까운 기록도 원인이라는 뜻은 아닙니다.
cal = maintenance.loc[maintenance.EQP_TAG.eq('P1-CR1-CAL01')].copy()
detection_inside = cal.DETC_DT.between(lot_start, lot_end)
work_overlaps = cal.STRT_DT.le(lot_end) & cal.END_DT.ge(lot_start)
related = cal.loc[detection_inside | work_overlaps].copy()
previous = cal.loc[cal.DETC_DT.lt(lot_start)].sort_values('DETC_DT').tail(1)
following = cal.loc[cal.DETC_DT.gt(lot_end)].sort_values('DETC_DT').head(1)
previous_work = cal.loc[cal.END_DT.lt(lot_start)].sort_values('END_DT').tail(1)
following_work = cal.loc[cal.STRT_DT.gt(lot_end)].sort_values('STRT_DT').head(1)
context = pd.concat([previous, previous_work, related, following, following_work])
context = context.drop_duplicates('WO_NO').sort_values('DETC_DT')
maintenance_names = {
    'TBM': '정기점검(TBM)',
    'BM': '고장 후 정비(BM)',
    'CBM': '상태기반 정비(CBM)',
}
change_names = part_changes.merge(
    parts[['PART_CD', 'PART_NM']],
    on='PART_CD',
    how='left',
    validate='many_to_one',
)
context_rows = []
for _, record in context.iterrows():
    if pd.isna(record.DETC_DT):
        relation = '감지 시각 없음'
    elif record.DETC_DT < lot_start:
        relation = f'LOT 시작 {((lot_start-record.DETC_DT).total_seconds()/3600):.2f}시간 전 감지'
    elif record.DETC_DT > lot_end:
        relation = f'LOT 종료 {((record.DETC_DT-lot_end).total_seconds()/3600):.2f}시간 후 감지'
    else:
        relation = '관측 중 감지'
    if record.STRT_DT <= lot_end and record.END_DT >= lot_start:
        relation += '; 작업 시간도 관측과 겹침'
    elif record.END_DT < lot_start:
        relation += f'; 작업 끝은 시작 {(lot_start-record.END_DT).total_seconds()/3600:.2f}시간 전'
    elif record.STRT_DT > lot_end:
        relation += f'; 작업 시작은 종료 {(record.STRT_DT-lot_end).total_seconds()/3600:.2f}시간 후'
    changes = change_names.loc[change_names.WO_NO.eq(record.WO_NO)]
    items = []
    for _, change in changes.iterrows():
        items.append(f'{change.PART_NM} {change.QTY}개({change.CHG_RSN})')
    context_rows.append({
        '작업번호': record.WO_NO,
        '유형': maintenance_names.get(record.MNT_TYPE, record.MNT_TYPE),
        '감지': record.DETC_DT,
        '작업 시작': record.STRT_DT,
        '작업 끝': record.END_DT,
        '관측과의 관계': relation,
        '기록 내용': record.REMARK if pd.notna(record.REMARK) else '기재 없음',
        '교체': ', '.join(items) if items else '교체 행 없음',
    })
context_table = pd.DataFrame(context_rows)
lot_events = events.loc[events.PLANT_CD.eq('CR1') & events.EVT_DT.between(lot_start, lot_end)]
event_table = lot_events[['EVT_ID', 'EVT_DT', 'EVT_TYPE', 'BEF_VAL', 'AFT_VAL']].copy()
cal_repairs = repair.loc[repair.PLANT_CD.eq('CR1')].copy()
# 종료 당일을 포함합니다. 관측 중 겹침과 가까운 계획을 구분합니다.
repair_overlaps = cal_repairs.STRT_DT.le(lot_end) & (cal_repairs.END_DT+pd.Timedelta(days=1)).gt(lot_start)

markers = []
for _, record in related.iterrows():
    if lot_start <= record.DETC_DT <= lot_end:
        label = maintenance_names.get(record.MNT_TYPE, record.MNT_TYPE)
        markers.append((record.DETC_DT, f'{label} 감지', '#a22b45'))

## 질문에 답하기 위한 실제 수치

| 확인 항목 | 재계산 결과 | 해석할 때 주의 |
| --- | --- | --- |
| 가열 4센서 평균: 시작→60분 뒤 | 815.915 → 850.537 ℃ (차이 34.623℃) | 정확한 안정화 시점의 정의가 아님 |
| 90분 뒤 가열 평균 | 858.872 ℃ | 이후에도 각 센서를 별도로 확인 |
| 90분 뒤부터 OP-Z4 중앙값 | 54.098 %; 다른 SPHC 044 41.739, 069 42.178, 091 44.359 | 평균 가동 부하를 통제한 비교 아님 |
| CP-Z4 중앙값 / OP-Z3 유효 분 | 0.449998 / 222분 | 센서별 관측≥30 조건 적용 |

**관찰:** 여러 온도가 관측 초기에 함께 오른다. 이후 온도가 약 860℃에 가까워져도 OP-Z4는 다른 SPHC LOT와 다른 수준을 유지한다. CP는 약 0.45를 중심으로 움직인다.

**가능한 해석과 한계:** 전날 정비 종료 뒤 재가열했을 가능성은 있지만, 제공된 조업이벤트만으로 재가동 명령을 확인할 수 없다. 높아진 출력만으로 설비 이상을 확정할 수 없다. 같은 강종에도 초기 운전 상태가 다르다.

In [ ]:
def median(lot_no, channel, start=None, end=None):
    data = minute_by_lot[lot_no]
    if start is not None:
        data = data.loc[data.index >= pd.Timestamp(start)]
    if end is not None:
        data = data.loc[data.index < pd.Timestamp(end)]
    return data[channel].median()

def at(channel, timestamp):
    return minute.loc[pd.Timestamp(timestamp), channel]

def raw_extreme(channel, maximum=True):
    series = raw_lot[channel].dropna()
    timestamp = series.idxmax() if maximum else series.idxmin()
    return timestamp, series.loc[timestamp]

def persistent_start(condition):
    # 유효 분만 넘겨 받아 마지막 유효 분까지 유지된 조건을 찾습니다.
    # 관측 종료 뒤는 알 수 없으며, 실시간 판단 규칙이 아닙니다.
    condition = condition.fillna(False)
    last_run = condition.iloc[::-1].cummin().iloc[::-1]
    times = last_run.index[last_run]
    return times[0] if len(times) else pd.NaT

start_heat = minute['HEAT_MEAN'].dropna().iloc[0]
heat60 = at('HEAT_MEAN', '2024-06-15 02:04')
heat90 = at('HEAT_MEAN', '2024-06-15 02:34')
after90 = minute.loc['2024-06-15 02:34':]
evidence_rows = [
    ('가열 4센서 평균: 시작→60분 뒤', f'{start_heat:.3f} → {heat60:.3f} ℃ (차이 {heat60-start_heat:.3f}℃)', '정확한 안정화 시점의 정의가 아님'),
    ('90분 뒤 가열 평균', f'{heat90:.3f} ℃', '이후에도 각 센서를 별도로 확인'),
    (
        '90분 뒤부터 OP-Z4 중앙값',
        (
            f"{after90['OP-Z4'].median():.3f} %; 다른 SPHC 044 {median(240044, 'OP-Z4'):.3f}"
            f", 069 {median(240069, 'OP-Z4'):.3f}, 091 {median(240091, 'OP-Z4'):.3f}"
        ),
        '평균 가동 부하를 통제한 비교 아님',
    ),
    ('CP-Z4 중앙값 / OP-Z3 유효 분', f"{minute['CP-Z4'].median():.6f} / {minute['OP-Z3'].notna().sum()}분", '센서별 관측≥30 조건 적용'),
]
markers.append((pd.Timestamp('2024-06-15 02:04'), '시작+60분 검토점', '#b77725'))

evidence_table = pd.DataFrame(evidence_rows, columns=['확인 항목', '재계산 결과', '해석할 때 주의'])
display(evidence_table)
for title, table in extra_tables:
    print(title)
    display(readable_table(table))

## 다른 센서에서도 확인한 변화

초기 온도 상승이 가열 구역에만 있는지 확인하기 위해 과시효 구역도 비교한다. 출력은 온도와 같은 방향으로 변해야 하는 값이 아니며, 이 변화의 끝을 자동 안정화 시각으로 삼지 않는다.

| 센서 | 첫 15분 유효 분 | 끝 15분 유효 분 | 첫 15분 중앙값 | 끝 15분 중앙값 | 끝−처음 | 끝 15분 최저 | 끝 15분 최고 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| TC-Z7 | 15 | 14 | 317.305 | 329.8 | 12.495 | 329.693 | 329.89 |
| TC-Z8 | 15 | 14 | 317.27 | 329.73 | 12.46 | 329.693 | 329.76 |
| OP-Z7 | 15 | 14 | 74.955 | 70.081 | -4.874 | 68.449 | 72.053 |
| OP-Z8 | 15 | 14 | 65.904 | 61.937 | -3.967 | 61.651 | 62.305 |

온도는 ℃, 출력은 % 가정이다. 첫·끝 15분은 시간 칸 기준이며 센서 결측으로 유효 분 수가 다를 수 있다.

In [ ]:
RELATED_SENSORS = ['TC-Z7', 'TC-Z8', 'OP-Z7', 'OP-Z8']

related_rows = []
for sensor in RELATED_SENSORS:
    values = minute[sensor]
    first = values.iloc[:15].dropna()
    last = values.iloc[-15:].dropna()
    related_rows.append({
        '센서': sensor,
        '첫 15분 유효 분': len(first),
        '끝 15분 유효 분': len(last),
        '첫 15분 중앙값': first.median(),
        '끝 15분 중앙값': last.median(),
        '끝−처음': last.median() - first.median(),
        '끝 15분 최저': last.min(),
        '끝 15분 최고': last.max(),
    })
related_table = pd.DataFrame(related_rows)
display(readable_table(related_table))

## 전체 관측에서 이 현상이 차지하는 위치

여러 온도가 관측 초기에 함께 오른다. 이후 온도가 약 860℃에 가까워져도 OP-Z4는 다른 SPHC LOT와 다른 수준을 유지한다. CP는 약 0.45를 중심으로 움직인다.

옅은 선은 원시 관측, 진한 선은 유효 1분 평균이다. 노란 영역은 다음 그림에서 확대할 범위이며 학습 제외 표시가 아니다. 다른 단위의 센서는 축을 분리했다.

![전체 관측](../../figures/07_lot_anomaly_review_namelesspark/07_LOT_240061/01_overview.png)

In [ ]:
def sensor_unit(sensor):
    if sensor.startswith('CP-'):
        return '단위 미확인'
    if sensor.startswith('OP-'):
        return '% (가정)'
    return '℃ (가정)'

def draw_timeseries(start, end, filename, title, overview=False):
    start = pd.Timestamp(start)
    end = pd.Timestamp(end)
    fig, axes = plt.subplots(len(FOCUS), 1, figsize=(12, 2.05*len(FOCUS)+1.0), sharex=True)
    axes = np.atleast_1d(axes)
    source = raw_lot.loc[start:end]
    averages = minute.loc[start.floor('min'):end.ceil('min')]
    # 없는 초를 NaN으로 둬 원시 관측 공백도 선으로 이어 붙이지 않습니다.
    grid = pd.date_range(start.ceil('s'), end.floor('s'), freq='1s')
    source = source.reindex(grid)
    for ax, sensor in zip(axes, FOCUS):
        ax.plot(source.index, source[sensor], color='#9bbbd1', lw=.6, alpha=.55, label='원시 관측')
        # 표의 13:29는 13:29:00~59 평균입니다. 곡선은 그 분의 가운데에 표시합니다.
        average_times = averages.index + pd.Timedelta(seconds=30)
        ax.plot(average_times, averages[sensor], color='#153f69', lw=1.2, label='1분 평균 (분 가운데, 관측≥30)')
        if overview:
            ax.axvspan(pd.Timestamp(ZOOM_START), pd.Timestamp(ZOOM_END), color='#e3bd65', alpha=.18, label='아래에서 확대할 구간')
        for timestamp, label, color in markers:
            if '1분 최고' in label:
                timestamp = timestamp + pd.Timedelta(seconds=30)
            if start <= timestamp <= end:
                ax.axvline(timestamp, color=color, lw=.9, ls='--', alpha=.85, label=label)
        for _, record in related.iterrows():
            left = max(start, record.STRT_DT)
            right = min(end, record.END_DT)
            if left < right:
                label = maintenance_names.get(record.MNT_TYPE, record.MNT_TYPE)
                ax.axvspan(left, right, color='#b5a5cc', alpha=.13, label=f'{label} 작업')
        ax.set_ylabel(f'{sensor}\n{sensor_unit(sensor)}', fontsize=10)
        ax.tick_params(labelsize=9)
        ax.grid(alpha=.18, lw=.5)
        ax.set_xlim(start, end)
    handles, labels = axes[0].get_legend_handles_labels()
    unique = {}
    for handle, label in zip(handles, labels):
        if label not in unique:
            unique[label] = handle
    fig.legend(unique.values(), unique.keys(), loc='upper center', bbox_to_anchor=(.5, .955), ncol=3, fontsize=9, frameon=False)
    locator = mdates.AutoDateLocator(minticks=4, maxticks=9)
    axes[-1].xaxis.set_major_locator(locator)
    axes[-1].xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))
    axes[-1].set_xlabel('관측 시각 — 색칠은 검토 범위이며 자동 학습 제외 표시가 아닙니다.', fontsize=9)
    fig.suptitle(f'LOT {LOT} · {grade} | {PLOT_TITLE}\n{title}', fontsize=13, y=.995)
    fig.subplots_adjust(left=.115, right=.98, bottom=.075, top=.89, hspace=.28)
    figure_path = Path(FIG_DIR) / filename
    fig.savefig(figure_path, dpi=160, facecolor='white')
    plt.show()
    plt.close(fig)
    return figure_path

def draw_comparison():
    if len(same_grade_lots) < 2:
        print(f'{grade}은 관측 LOT가 하나여서 같은 강종의 독립 비교가 불가능합니다.')
        return None
    fig, axes = plt.subplots(len(COMPARE), 1, figsize=(10, 3.0*len(COMPARE)), squeeze=False)
    labels = []
    for lot_no in same_grade_lots:
        label = str(lot_no)
        if grade == 'SPCEN' and lot_no == 240114:
            label += '\n20:10 하강 전'
        labels.append(label)
    for ax, sensor in zip(axes[:, 0], COMPARE):
        values = [comparison_values[lot_no][sensor].dropna().to_numpy() for lot_no in same_grade_lots]
        boxes = ax.boxplot(values, tick_labels=labels, showfliers=False, widths=.45, patch_artist=True)
        for patch, lot_no in zip(boxes['boxes'], same_grade_lots):
            patch.set_facecolor('#82afca' if lot_no == LOT else '#dce6ec')
        ax.set_ylabel(f'{sensor}\n{sensor_unit(sensor)}')
        ax.grid(axis='y', alpha=.2)
        ax.set_title(f'{sensor}: LOT별 분포 (상자 25~75%, 가운데 선 중앙값)', fontsize=10)
    axes[-1, 0].set_xlabel('LOT — 극단점 생략. 원시 최댓값은 표·시계열에서 확인하세요.\n서로 다른 날짜·관측 길이·운전 조건을 포함합니다.')
    fig.suptitle(f'{grade} {len(same_grade_lots)}개 LOT 비교 · 진한 상자 = {LOT}', fontsize=13)
    fig.tight_layout(rect=[0, 0, 1, .95])
    path = Path(FIG_DIR) / '03_same_grade.png'
    fig.savefig(path, dpi=160, facecolor='white')
    plt.show()
    plt.close(fig)
    return path

overview_path = draw_timeseries(lot_start, lot_end, '01_overview.png', '전체 관측과 검토 구간', overview=True)

## 원시값까지 확대해서 확인

확대 범위: **2024-06-15 01:04 ~ 2024-06-15 02:45**. 이 범위에 감지·작업 기록이 있으면 감지 시각은 점선, 작업 시간은 보라색으로 표시한다. 별도로 비교한 시각도 점선으로 표시하며 뜻은 범례에 적었다.

전날 정비 종료 뒤 재가열했을 가능성은 있지만, 제공된 조업이벤트만으로 재가동 명령을 확인할 수 없다. 높아진 출력만으로 설비 이상을 확정할 수 없다. 같은 강종에도 초기 운전 상태가 다르다.

![확대 관측](../../figures/07_lot_anomaly_review_namelesspark/07_LOT_240061/02_zoom.png)

In [ ]:
zoom_path = draw_timeseries(ZOOM_START, ZOOM_END, '02_zoom.png', '관심 구간의 원시 관측과 1분 평균')

## 추가 검토: 초기 상승 뒤에도 Z3 온도와 출력은 크게 왕복한다

초기 상승을 지난 뒤의 별도 변화를 확인한다. 아래 구간은 최신 05에서 초기 보류를 마치고 다시 남기는 시각인 02:31 이후다. 초기 보류를 줄였다는 사실이 남은 구간 전체의 안정성을 보증하지는 않는다.

| 확인 항목 | 시각 또는 구간 | 재계산 결과 |
| --- | --- | --- |
| OP-Z3 0 연속 기록 | 03:12:03~03:18:39 | 0; 원시 관측 389개 |
| OP-Z3 높은 값 연속 기록 | 03:22:15~03:28:44 | 94.2826; 원시 관측 383개 |
| TC-Z3 1분 평균 최고→이후 최저 | 03:15→03:26 | 870.570→850.742℃ (차이 -19.827℃) |

시각의 초가 있는 행은 원자료 기록이며, 온도 최고·최저는 관측≥30인 1분 평균이다. OP 단위 %는 가정이다.

061의 Z3 왕복은 069·091에서도 비슷하게 관측된다. 출력이 0으로 기록된 동안 온도가 오르고, 높은 출력이 반복되는 동안 온도가 낮아진다. 제어 반응인지 설비 문제인지는 이 자료만으로 확정할 수 없다. 05의 적용표는 그대로 두고 이 구간의 학습 영향을 따로 확인한다. 이번에는 구역 온도 상승·하강과 CP 범위 이탈을 먼저 보류하기로 선택했다. 출력 왕복은 별도 적용 기준을 정하지 않아 검토로 남겼다.


In [ ]:
SECONDARY_START = '2024-06-15 03:08'
SECONDARY_END = '2024-06-15 03:35'
SECONDARY_TITLE = '초기 상승 뒤에도 Z3 온도와 출력은 크게 왕복한다'
SECONDARY_KIND = 'output'
SECONDARY_SENSORS = ['TC-Z3', 'OP-Z3']

# 검토 시각은 관찰을 확대하기 위한 선택이며 자동 제외 규칙이 아닙니다.
secondary_start = pd.Timestamp(SECONDARY_START)
secondary_end = pd.Timestamp(SECONDARY_END)
secondary_raw = raw_lot.loc[secondary_start:secondary_end]
secondary_minute = minute.loc[secondary_start:secondary_end]

def longest_recorded_value(series, value):
    # 값이 같은 원시 관측이 연속해서 기록된 가장 긴 구간을 찾습니다.
    same = series.eq(value)
    run_ids = same.ne(same.shift()).cumsum()
    observations = pd.DataFrame({
        '시각': series.index,
        '같은값': same.to_numpy(),
        '구간': run_ids.to_numpy(),
    })
    groups = observations.loc[observations['같은값']].groupby('구간')
    summary = groups.agg(
        start=('시각', 'min'),
        end=('시각', 'max'),
        count=('시각', 'size'),
    )
    return summary.loc[summary['count'].idxmax()]

secondary_upper = secondary_raw['OP-Z3'].max()
secondary_zero_run = longest_recorded_value(secondary_raw['OP-Z3'], 0)
secondary_upper_run = longest_recorded_value(secondary_raw['OP-Z3'], secondary_upper)
secondary_peak_time = secondary_minute['TC-Z3'].idxmax()
secondary_low_time = secondary_minute.loc[secondary_peak_time:, 'TC-Z3'].idxmin()
secondary_peak = secondary_minute.loc[secondary_peak_time, 'TC-Z3']
secondary_low = secondary_minute.loc[secondary_low_time, 'TC-Z3']
secondary_rows = [
    {
        '확인 항목': 'OP-Z3 0 연속 기록',
        '시각 또는 구간': f"{secondary_zero_run.start:%H:%M:%S}~{secondary_zero_run.end:%H:%M:%S}",
        '재계산 결과': f"0; 원시 관측 {secondary_zero_run['count']}개",
    },
    {
        '확인 항목': 'OP-Z3 높은 값 연속 기록',
        '시각 또는 구간': f"{secondary_upper_run.start:%H:%M:%S}~{secondary_upper_run.end:%H:%M:%S}",
        '재계산 결과': f"{secondary_upper:.4f}; 원시 관측 {secondary_upper_run['count']}개",
    },
    {
        '확인 항목': 'TC-Z3 1분 평균 최고→이후 최저',
        '시각 또는 구간': f'{secondary_peak_time:%H:%M}→{secondary_low_time:%H:%M}',
        '재계산 결과': f'{secondary_peak:.3f}→{secondary_low:.3f}℃ (차이 {secondary_low-secondary_peak:.3f}℃)',
    },
]
secondary_points = {
    'TC-Z3': [secondary_peak_time, secondary_low_time],
}
secondary_change_time = None

secondary_table = pd.DataFrame(secondary_rows)
display(secondary_table.round(3))


In [ ]:
# 원시 공백은 선으로 잇지 않고, 평균은 분 가운데에 표시합니다.
secondary_grid = pd.date_range(secondary_start.ceil('s'), secondary_end.floor('s'), freq='1s')
secondary_plot_raw = raw_lot.loc[secondary_start:secondary_end].reindex(secondary_grid)
secondary_plot_minute = minute.loc[secondary_start.floor('min'):secondary_end.ceil('min')]
secondary_average_times = secondary_plot_minute.index + pd.Timedelta(seconds=30)
secondary_fig, secondary_axes = plt.subplots(
    len(SECONDARY_SENSORS),
    1,
    figsize=(12, 2.35 * len(SECONDARY_SENSORS) + 1.4),
    sharex=True,
)
for ax, sensor in zip(secondary_axes, SECONDARY_SENSORS):
    ax.plot(secondary_plot_raw.index, secondary_plot_raw[sensor], color='#9bbbd1', lw=0.7, alpha=0.65, label='원시 관측')
    ax.plot(secondary_average_times, secondary_plot_minute[sensor], color='#153f69', lw=1.25, label='1분 평균 (분 가운데, 관측≥30)')
    points = secondary_points.get(sensor, [])
    if points:
        point_times = pd.DatetimeIndex(points)
        point_values = minute.loc[point_times, sensor]
        ax.scatter(point_times + pd.Timedelta(seconds=30), point_values, color='#c76f2c', s=28, zorder=5, label='위 표의 비교점')
    if SECONDARY_KIND == 'output':
        ax.axvspan(secondary_zero_run.start, secondary_zero_run.end, color='#67a7a4', alpha=0.17, label='OP-Z3 0 연속 기록 구간')
        ax.axvspan(secondary_upper_run.start, secondary_upper_run.end, color='#e5ad57', alpha=0.2, label='OP-Z3 높은 값 연속 기록 구간')
        if secondary_change_time is not None:
            ax.axvline(secondary_change_time, color='#9564a7', lw=1, ls='--', label='이전 관측 상단을 처음 넘은 시각')
    else:
        ax.axvline(secondary_boundary, color='#9564a7', lw=1, ls='--', label=f'가열 온도 비교 기준 {secondary_boundary:%H:%M}')
    unit = '% (가정)' if sensor.startswith('OP-') else '℃ (가정)'
    ax.set_ylabel(f'{sensor}\n{unit}', fontsize=10)
    ax.set_xlim(secondary_start, secondary_end)
    ax.grid(alpha=0.2, lw=0.5)
    ax.tick_params(labelsize=9)

secondary_handles = {}
for ax in secondary_axes:
    handles, labels = ax.get_legend_handles_labels()
    for handle, label in zip(handles, labels):
        if label not in secondary_handles:
            secondary_handles[label] = handle
secondary_fig.legend(
    secondary_handles.values(),
    secondary_handles.keys(),
    loc='upper center',
    bbox_to_anchor=(0.5, 0.92),
    ncol=3,
    fontsize=8.5,
    frameon=False,
)
secondary_locator = mdates.AutoDateLocator(minticks=5, maxticks=9)
secondary_axes[-1].xaxis.set_major_locator(secondary_locator)
secondary_axes[-1].xaxis.set_major_formatter(mdates.ConciseDateFormatter(secondary_locator))
secondary_axes[-1].set_xlabel('관측 시각 — 점·음영·기준선은 비교용이며 자동 제외 표시가 아닙니다.', fontsize=9)
secondary_fig.suptitle(f'LOT {LOT} · {grade}\n{SECONDARY_TITLE}', fontsize=13, y=0.995)
secondary_fig.subplots_adjust(left=0.105, right=0.985, bottom=0.1, top=0.80, hspace=0.3)
secondary_path = Path(FIG_DIR) / '04_secondary_review.png'
secondary_fig.savefig(secondary_path, dpi=160, facecolor='white')
plt.show()
plt.close(secondary_fig)


![초기 상승 뒤에도 Z3 온도와 출력은 크게 왕복한다](../../figures/07_lot_anomaly_review_namelesspark/07_LOT_240061/04_secondary_review.png)

옅은 선은 원시 관측, 진한 선은 1분 평균이다. 표는 분 시작 시각을 쓰고 평균 곡선·비교점은 30초 뒤인 분 가운데에 표시한다. 관측 공백은 보간하지 않는다.

## 같은 강종 비교

같은 강종 **4개 LOT**: 240044, 240061, 240069, 240091. 각 LOT를 따로 비교하며, 긴 LOT에 가중된 강종 전체 평균은 만들지 않았다. 날짜·관측 길이·운전 상태·소재 부하가 다르므로 강종 효과만 분리한 비교는 아니다.

| LOT | 시작일 | 비교 범위 | 센서 | 유효 분 | 25% | 중앙값 | 75% |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 240044 | 2024-05-03 | 관측 전체 | TC-Z4 | 75 | 859.436 | 859.678 | 859.851 |
| 240044 | 2024-05-03 | 관측 전체 | OP-Z4 | 75 | 41.086 | 41.739 | 42.938 |
| 240061 | 2024-06-15 | 관측 전체 | TC-Z4 | 225 | 849.389 | 859.339 | 859.865 |
| 240061 | 2024-06-15 | 관측 전체 | OP-Z4 | 225 | 53.482 | 54.291 | 55.319 |
| 240069 | 2024-07-07 | 관측 전체 | TC-Z4 | 813 | 859.428 | 859.655 | 859.841 |
| 240069 | 2024-07-07 | 관측 전체 | OP-Z4 | 813 | 40.749 | 42.178 | 43.83 |
| 240091 | 2024-09-08 | 관측 전체 | TC-Z4 | 641 | 859.502 | 859.699 | 859.934 |
| 240091 | 2024-09-08 | 관측 전체 | OP-Z4 | 641 | 43.224 | 44.359 | 45.233 |

분포 그림은 극단점을 생략했다. 최대·최소와 짧은 급변은 위 수치표와 시계열에서 확인한다.

![같은 강종 분포](../../figures/07_lot_anomaly_review_namelesspark/07_LOT_240061/03_same_grade.png)

In [ ]:
display(readable_table(comparison_table))
comparison_path = draw_comparison()

## 조업·정비 기록은 무엇을 뒷받침하는가

CAL01의 관측 중 감지·작업 중첩과 가장 가까운 이전/이후 감지·작업 기록을 구분했다. 가까운 기록을 자동 원인으로 연결하지 않는다. 정기점검(TBM), 고장 후 정비(BM), 상태기반 정비(CBM)는 기록의 유형이다. 정비이력의 감지 시각은 실제 이상 시작 시각과 같다고 볼 수 없으며, 특히 정기점검(TBM)의 감지를 고장 발생으로 해석하지 않는다. 정기수리 캘린더는 계획 기간이며, 실제 작업 시각은 별도의 정비이력과 구분한다.

| 작업번호 | 유형 | 감지 | 작업 시작 | 작업 끝 | 관측과의 관계 | 기록 내용 | 교체 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| WO-2024-00972 | 정기점검(TBM) | 2024-06-14 14:09:00 | 2024-06-14 15:35:00 | 2024-06-14 18:03:00 | LOT 시작 10.92시간 전 감지; 작업 끝은 시작 7.02시간 전 | 1호기 재발, SD작업 수행 | 교체 행 없음 |
| WO-2024-01094 | 고장 후 정비(BM) | 2024-07-08 08:49:00 | 2024-07-08 11:11:00 | 2024-07-08 19:04:00 | LOT 종료 555.99시간 후 감지; 작업 시작은 종료 558.36시간 후 | 1호기 재발, 절연 저하 | O링 세트 3개(예방) |

관측 안에 있는 CR1 조업이벤트:

| EVT_ID | EVT_DT | EVT_TYPE | BEF_VAL | AFT_VAL |
| --- | --- | --- | --- | --- |
| EVT-2024-0122 | 2024-06-15 01:04:00 | 강종변경 | DP590 | SPHC |

이 CSV의 CR1 이벤트 유형은 강종변경이다. 이 기록만으로 출력 설정, 재가열·정지 명령, 소재 이동 속도를 확인할 수 없다.

관측과 겹치는 CR1 정기수리 계획은 **0건**이다. 아래 계획의 종료일은 당일까지 포함한다. 계획이 가깝다는 사실과 센서 변화 원인은 구분한다.

| PLAN_ID | STRT_DT | END_DT | SCOPE |
| --- | --- | --- | --- |
| PLAN-2024-02 | 2024-06-10 00:00:00 | 2024-06-14 00:00:00 | 연속소둔로 노내 정비 |
| PLAN-2024-04 | 2024-11-05 00:00:00 | 2024-11-10 00:00:00 | 냉연 라인 전 설비 정기수리 |

In [ ]:
display(context_table)
display(event_table)
print('관측과 겹친 CR1 정기수리 계획:', int(repair_overlaps.sum()), '건')
display(cal_repairs[['PLAN_ID', 'STRT_DT', 'END_DT', 'SCOPE']])

## 학습에 넣을지에 대한 현재 판단

기존 218분 제외를 철회한다. 최신 05는 마지막 30분 안의 유효 구역 온도 평균 중앙값에서 **±2℃ 이내로 20분 연속 머문 첫 구간의 시작**부터 남기는 사후 선택을 사용한다. 가열 4개 온도 평균은 02:31부터, 과시효 2개 온도 평균은 02:15부터 이 조건에 들어온다. 따라서 앞부분의 보류 범위는 각각 87분과 71분이다. 20분이 지났음을 확인할 수 있는 시각은 각각 02:51·02:35이며, 그때까지 추가로 버리는 것은 아니다.

가열 8개는 `TC-Z3`~`TC-Z6`·`OP-Z3`~`OP-Z6`, 과시효 4개는 `TC-Z7`·`TC-Z8`·`OP-Z7`·`OP-Z8`이다. 초기·후반 출력 수준도 달라 온도와 같은 운전 단계 표시를 적용했다. `CP-Z4`는 유지하고 `TC-Z9`는 별도 검토한다. 첫 60분의 약 35℃ 상승은 관측 요약이며 이 제외 규칙 자체가 아니다. 시험 LOT 240117에 이 종료 시각을 복사하거나 시험 자료로 선택값을 조정하지 않는다.

**아래 시간은 시작·끝이 포함되는 1분 칸 전체**를 뜻한다. 예를 들어 04:59~05:00은 04:59:00 이상 05:01:00 미만이다. 초 단위 원자료 근거와 사후 경계는 05 적용표에서 구분한다. 원본 행을 지우지 않고 대상 센서에만 표시하며, 고장 정답이 아니다.

| 05 구간 | 대상 | 표시하는 1분 범위 | 실제 유효 보류분 |
| --- | --- | --- | --- |
| A010 | 가열 8개 | 06/15 01:04 ~ 06/15 02:30 | 센서당 87분 (`OP-Z3` 84분) |
| A011 | 과시효 4개 | 06/15 01:04 ~ 06/15 02:14 | 센서당 71분 |

관찰된 차이, 가능한 원인, 모델 입력·학습 판단은 서로 구분한다. 이 노트북의 색칠과 감지선은 정답 라벨이 아니다. 단일 LOT·강종별 소수 LOT·간헐 관측이라는 한계를 유지한다.

[전체 LOT 해석(노션)](https://app.notion.com/p/3f7e609a45d080b88934e41f4403a466) · [05 학습 제외 기준](../05_exclusion_windows_namelesspark.ipynb)